In [1]:
"""
00_database_setup.py
--------------------
Loads the processed CSV files into a SQLite database (weather.db).

Run this script ONCE before running any other notebook.

Pipeline:
    data/processed/*.csv  ->  SQLite (weather.db)  ->  notebooks 01, 02, 03...

Usage:
    python src/00_database_setup.py
"""

import sqlite3
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------------
DATA_DIR = Path("../data/processed")
DB_PATH  = DATA_DIR / "weather.db"

print(f"Data folder : {DATA_DIR}")
print(f"Database    : {DB_PATH}")

# ------------------------------------------------------------------
# 2. Load CSVs
# ------------------------------------------------------------------
print("\nLoading CSV files...")

df_details    = pd.read_csv(DATA_DIR / "details_consolidated.csv", low_memory=False)
df_fatalities = pd.read_csv(DATA_DIR / "fatalities_consolidated.csv", low_memory=False)

# --- ADICIONE ESTA PARTE AQUI ---
print("Filtering non-mainland states...")

non_mainland = [
    'ALASKA', 'HAWAII', 'PUERTO RICO', 'AMERICAN SAMOA', 
    'GUAM', 'VIRGIN ISLANDS', 'NORTHERN MARIANA ISLANDS'
]

# Filtramos apenas o df_details, pois ele contém a coluna STATE
df_details = df_details[~df_details['STATE'].str.upper().isin(non_mainland)].copy()

# Opcional: Filtrar o df_fatalities para manter apenas fatalidades de eventos que restaram
# Isso garante integridade referencial entre as tabelas
df_fatalities = df_fatalities[df_fatalities['EVENT_ID'].isin(df_details['EVENT_ID'])].copy()
# --------------------------------

print(f"  details     : {df_details.shape[0]:,} rows x {df_details.shape[1]} cols (Mainland only)")
print(f"  fatalities  : {df_fatalities.shape[0]:,} rows x {df_fatalities.shape[1]} cols")

# ------------------------------------------------------------------
# 3. Write to SQLite
# ------------------------------------------------------------------
print("\nWriting to database...")

conn = sqlite3.connect(DB_PATH)

df_details.to_sql("details", conn, if_exists="replace", index=False)
print("  Table 'details' created.")

df_fatalities.to_sql("fatalities", conn, if_exists="replace", index=False)
print("  Table 'fatalities' created.")

conn.close()
print(f"\nDone. Database saved to: {DB_PATH}")

# ------------------------------------------------------------------
# 4. Verify — row counts
# ------------------------------------------------------------------
print("\nVerifying...")

conn = sqlite3.connect(DB_PATH)

for table in ["details", "fatalities"]:
    count = pd.read_sql_query(f"SELECT COUNT(*) AS rows FROM {table};", conn)
    print(f"  {table}: {count['rows'][0]:,} rows")

conn.close()

# ------------------------------------------------------------------
# 5. Example queries — sanity check
# ------------------------------------------------------------------
print("\n--- Top 10 event types ---")
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("""
    SELECT EVENT_TYPE,
           COUNT(*) AS total_events
    FROM details
    GROUP BY EVENT_TYPE
    ORDER BY total_events DESC
    LIMIT 10;
""", conn)

print(df.to_string(index=False))

print("\n--- Events per year ---")
df = pd.read_sql_query("""
    SELECT YEAR,
           COUNT(*) AS total_events
    FROM details
    GROUP BY YEAR
    ORDER BY YEAR;
""", conn)

print(df.to_string(index=False))

print("\n--- Top 10 event types by fatalities ---")
df = pd.read_sql_query("""
    SELECT d.EVENT_TYPE,
           COUNT(f.FATALITY_ID) AS total_fatalities
    FROM details d
    LEFT JOIN fatalities f ON d.EVENT_ID = f.EVENT_ID
    GROUP BY d.EVENT_TYPE
    ORDER BY total_fatalities DESC
    LIMIT 10;
""", conn)

print(df.to_string(index=False))

conn.close()

print("\nSetup complete. You can now use database.py in your notebooks.")

Data folder : ..\data\processed
Database    : ..\data\processed\weather.db

Loading CSV files...
Filtering non-mainland states...
  details     : 1,491,990 rows x 51 cols (Mainland only)
  fatalities  : 19,590 rows x 11 cols

Writing to database...
  Table 'details' created.
  Table 'fatalities' created.

Done. Database saved to: ..\data\processed\weather.db

Verifying...
  details: 1,491,990 rows
  fatalities: 19,590 rows

--- Top 10 event types ---
              EVENT_TYPE  total_events
       Thunderstorm Wind        384590
                    Hail        286401
             Flash Flood         90013
               High Wind         76789
            Winter Storm         75348
          Winter Weather         75020
                 Drought         70786
              Heavy Snow         60246
                   Flood         57664
Marine Thunderstorm Wind         40894

--- Events per year ---
 YEAR  total_events
 2000         50962
 2001         47672
 2002         49993
 2003      